# Etapa 3 — Diagnóstico e robustez da primeira MLP

Modelo congelado para BOs envolvendo celulares subtraídos da capital. 0=FURTO, 1=ROUBO. Não prevê crimes futuros. Somente 2023/2024; 2025 permanece sem avaliação. O notebook lê agregados dos scripts reproduzíveis, sem repetir treinos automaticamente.

In [1]:
from pathlib import Path
import json,csv
ROOT=Path.cwd()
if not (ROOT/'reports').exists():ROOT=ROOT.parent
OUT=ROOT/'reports/etapa3'
def show(name,attribute=None):
    with (OUT/(name+'.csv')).open(encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if attribute is None or row.get('atributo')==attribute:print(row)
print(json.loads((OUT/'resumo.json').read_text(encoding='utf-8')))


{'validacao': {'n': 158895, 'furtos': 92055, 'roubos': 66840, 'tn': 68328, 'fp': 23727, 'fn': 25962, 'tp': 40878, 'accuracy': 0.6872840555083546, 'macro_f1': 0.6776641034482787, 'balanced_accuracy': 0.6769159034481669, 'precision_roubo': 0.6327374042256791, 'recall_roubo': 0.6115798922800718, 'f1_roubo': 0.6219787743923314}, 'architecture': {'input': 299, 'units': [64, 1], 'params': 19265}, 'threshold': 0.5, 'years': [2023, 2024], 'ece_10': 0.020581202021013947, 'ece_20': 0.020828606930706053, 'brier': 0.2009063087680951, 'log_loss': 0.5846715435989691, 'prob_roubo_media': 0.4358602464199066, 'prevalencia_roubo': 0.42065514962711226, 'brier_constante_prevalencia_treino': 0.24408854027279436, 'permutation': {'sample_size': 30000, 'sampling_seed': 314159, 'repeats': 5, 'same_sample_baseline': {'n': 30000, 'furtos': 17373, 'roubos': 12627, 'tn': 12949, 'fp': 4424, 'fn': 4982, 'tp': 7645, 'accuracy': 0.6864666666666667, 'macro_f1': 0.6763499352672344, 'balanced_accuracy': 0.675400312380693

## Erros e condições de bairro

Matriz reproduzida: TN=68.328, FP=23.727, FN=25.962, TP=40.878. Destaques exigem n≥200 e pelo menos 30 exemplos por classe. Taxas altas com prevalências diferentes não são diretamente comparáveis. Bairros não são classificados em níveis de segurança.

In [2]:
show('metricas_grupos','condicao_bairro')
show('metricas_grupos','presenca_local')


{'atributo': 'condicao_bairro', 'categoria': 'ausente', 'n': '521', 'furtos': '167', 'roubos': '354', 'tn': '60', 'fp': '107', 'fn': '38', 'tp': '316', 'accuracy': '0.7216890595009597', 'macro_f1': '0.6331075010320293', 'balanced_accuracy': '0.6259684021786934', 'precision_roubo': '0.7470449172576832', 'recall_roubo': '0.8926553672316384', 'f1_roubo': '0.8133848133848134', 'taxa_erro': '0.2783109404990403', 'erro_ic95_inf': '0.24155727284741133', 'erro_ic95_sup': '0.3183098144077543', 'fpr': '0.6407185628742516', 'fnr': '0.10734463276836158', 'prob_roubo_media': '0.6659017205238342', 'prob_media_FN': '0.3155559301376343', 'prob_media_FP': '0.7349655032157898', 'destacar': 'True'}
{'atributo': 'condicao_bairro', 'categoria': 'desconhecido', 'n': '355', 'furtos': '130', 'roubos': '225', 'tn': '38', 'fp': '92', 'fn': '11', 'tp': '214', 'accuracy': '0.7098591549295775', 'macro_f1': '0.6153036854674957', 'balanced_accuracy': '0.6217094017094017', 'precision_roubo': '0.6993464052287581', 're

![Erros por classe](../reports/etapa3/figures/erros_por_classe.png)

## Generalização

A ausência de local se associa ao rótulo em ambos os anos; OUTROS muda de 12 para 7.706 registros. Isso é sinal para investigar preenchimento ou recodificação, não prova de vazamento. A piora de val_loss após a época 5 é compatível com sobreajuste incremental, mas o drift também contribui ao gap. Não há evidência suficiente para concluir underfitting por capacidade.

In [3]:
show('ausencias_ano_classe','tipo_local')
show('drift_atributos')
show('ambiguidade_contextos')


{'ano': '2023', 'atributo': 'tipo_local', 'classe': '0', 'n': '106421', 'ausentes': '12613', 'taxa_ausencia': '0.11851984100882344'}
{'ano': '2023', 'atributo': 'tipo_local', 'classe': '1', 'n': '83703', 'ausentes': '1418', 'taxa_ausencia': '0.01694085038768025'}
{'ano': '2024', 'atributo': 'tipo_local', 'classe': '0', 'n': '92055', 'ausentes': '4174', 'taxa_ausencia': '0.04534245831296507'}
{'ano': '2024', 'atributo': 'tipo_local', 'classe': '1', 'n': '66840', 'ausentes': '479', 'taxa_ausencia': '0.007166367444643926'}
{'atributo': 'mes', 'distancia_variacao_total': '0.01812989892246382', 'categorias_2023': '12', 'categorias_2024': '12', 'novas_2024': '0', 'registros_novos_2024': '0'}
{'atributo': 'dia_semana', 'distancia_variacao_total': '0.007601141166298925', 'categorias_2023': '7', 'categorias_2024': '7', 'novas_2024': '0', 'registros_novos_2024': '0'}
{'atributo': 'tipo_local', 'distancia_variacao_total': '0.057568487175362525', 'categorias_2023': '29', 'categorias_2024': '29', '

![Métricas por mês](../reports/etapa3/figures/desempenho_temporal.png)

## Influência preditiva

Permutação de atributos originais, amostra fixa de 30 mil BOs de 2024, cinco repetições. Bairro e local dominam. Importância marginal quebra dependências e não mede causalidade; DP não é IC populacional. Nenhum atributo foi removido.

In [4]:
show('importancia_atributos')


{'atributo': 'mes', 'delta_macro_f1_media': '0.00012156612196563276', 'delta_balanced_accuracy_media': '-6.6285612601246765e-06', 'delta_accuracy_media': '0.00043333333333333', 'delta_macro_f1_dp': '0.0011696679743555998'}
{'atributo': 'dia_semana', 'delta_macro_f1_media': '0.0022133274689665862', 'delta_balanced_accuracy_media': '0.0018981508091912014', 'delta_accuracy_media': '0.0028799999999999937', 'delta_macro_f1_dp': '0.0011016497790114251'}
{'atributo': 'tipo_local', 'delta_macro_f1_media': '0.06472890262016365', 'delta_balanced_accuracy_media': '0.06460156144943649', 'delta_accuracy_media': '0.05888000000000002', 'delta_macro_f1_dp': '0.0018378391766285932'}
{'atributo': 'bairro', 'delta_macro_f1_media': '0.11664847776174299', 'delta_balanced_accuracy_media': '0.11574902530268305', 'delta_accuracy_media': '0.10946666666666667', 'delta_macro_f1_dp': '0.003197507886261515'}


![Importância](../reports/etapa3/figures/importancia_atributos.png)

## Probabilidades

Brier e Log Loss combinam discriminação e confiabilidade. ECE usa 10 e 20 bins de largura uniforme, com último incluindo 1. ECE global pode esconder problemas de subgrupos. Não foi ajustado calibrador, e o threshold permanece 0,5.

In [5]:
show('confianca')


{'grupo': 'todos', 'n': '158895', 'confianca_media': '0.6982601881027222', 'prob_classe_verdadeira_media': '0.6033074855804443', 'mediana_prob_roubo': '0.43421733379364014', 'n_confianca_ge_0.8': '34732', 'n_confianca_ge_0.9': '13105', 'n_confianca_ge_0.95': '2843'}
{'grupo': 'corretos', 'n': '109206', 'confianca_media': '0.7193909287452698', 'prob_classe_verdadeira_media': '0.7193909287452698', 'mediana_prob_roubo': '0.39149388670921326', 'n_confianca_ge_0.8': '30358', 'n_confianca_ge_0.9': '12226', 'n_confianca_ge_0.95': '2725'}
{'grupo': 'incorretos', 'n': '49689', 'confianca_media': '0.65181964635849', 'prob_classe_verdadeira_media': '0.3481804430484772', 'mediana_prob_roubo': '0.4884721636772156', 'n_confianca_ge_0.8': '4374', 'n_confianca_ge_0.9': '879', 'n_confianca_ge_0.95': '118'}
{'grupo': 'FP', 'n': '23727', 'confianca_media': '0.643943727016449', 'prob_classe_verdadeira_media': '0.356056272983551', 'mediana_prob_roubo': '0.6453821659088135', 'n_confianca_ge_0.8': '1073', 'n

![Confiança](../reports/etapa3/figures/distribuicao_confianca.png)
![Calibração](../reports/etapa3/figures/curva_calibracao.png)
![Limiares](../reports/etapa3/figures/precision_recall_limiares.png)

## Robustez e estabilidade

Caixa/espaços e aliases de bairro são consistentes, mas local sem acento pode cair no fallback de ausente. Mês/dia None não tiveram exemplos de treino. Essas limitações foram documentadas sem modificar o baseline. Controle 42 e duas sementes adicionais mantêm arquitetura, hiperparâmetros e pré-processador. Nenhum vencedor foi escolhido.

In [6]:
show('estabilidade_sementes')
show('sondagens_inferencia')
print((OUT/'testes.json').read_text(encoding='utf-8'))


{'seed': '42', 'batch_order_seed': '42', 'epochs': '10', 'best_epoch': '5', 'seconds': '20.374827599996934', 'n': '158895', 'furtos': '92055', 'roubos': '66840', 'tn': '68328', 'fp': '23727', 'fn': '25962', 'tp': '40878', 'accuracy': '0.6872840555083546', 'macro_f1': '0.6776641034482787', 'balanced_accuracy': '0.6769159034481669', 'precision_roubo': '0.6327374042256791', 'recall_roubo': '0.6115798922800718', 'f1_roubo': '0.6219787743923314', 'baseline_max_abs_diff': '0.0', 'baseline_reproduced_atol_1e7': 'True'}
{'seed': '7', 'batch_order_seed': '42', 'epochs': '10', 'best_epoch': '5', 'seconds': '17.368538100010483', 'n': '158895', 'furtos': '92055', 'roubos': '66840', 'tn': '68433', 'fp': '23622', 'fn': '26121', 'tp': '40719', 'accuracy': '0.6869442084395355', 'macro_f1': '0.6771218303984325', 'balanced_accuracy': '0.6762968071347558', 'precision_roubo': '0.6328624049983681', 'recall_roubo': '0.6092010771992818', 'f1_roubo': '0.6208063667756764', 'baseline_max_abs_diff': '', 'baselin

## Recomendações e limites

Priorizar ablação controlada de tipo de local, raridade de bairros, validação temporal e contrato da inferência. Comparar regularização com hipótese de sobreajuste, sem presumir ganhos por aumentar camadas. Reservar dados próprios para calibração/threshold; os mesmos dados não podem avaliar imparcialmente a escolha. Ver `../docs/recomendacoes_etapa4.md` e `../reports/relatorio_etapa3.md`. Nenhuma Etapa 4 foi iniciada.